# 02 — Unsupervised Clustering

KMeans and HDBSCAN over each cached embedding method (UMAP-reduced first),
scored against the hidden ground-truth labels via the Hungarian-matched
accuracy and standard clustering metrics.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

import json

import hdbscan
import pandas as pd
import umap
from sklearn.cluster import KMeans

from utils import config
from utils.data import stratified_sample
from utils.embeddings import load_cached
from utils.metrics import evaluate_unsupervised

In [2]:
train_clean = pd.read_parquet(config.PROCESSED_DIR / "train_clean.parquet")

train_sample = stratified_sample(train_clean, config.SAMPLE_SIZE, seed=config.SEED)
suffix = f"n{config.SAMPLE_SIZE}" if config.SAMPLE_SIZE else "full"
true_labels = train_sample["label"].to_numpy()

# RoBERTa was embedded on a separate, smaller sample (see Task 7) — its
# true_labels must come from that same sample, not train_sample above.
roberta_train_sample = stratified_sample(train_clean, config.ROBERTA_SAMPLE_SIZE, seed=config.SEED)
roberta_suffix = f"n{config.ROBERTA_SAMPLE_SIZE}" if config.ROBERTA_SAMPLE_SIZE else "full"
roberta_true_labels = roberta_train_sample["label"].to_numpy()

METHODS = ["tfidf", "minilm", "roberta"]
if config.OPENAI_API_KEY and load_cached(f"openai_train_{suffix}") is not None:
    # Gate on cache existence, not just key presence — the OpenAI embedding
    # cell in 01_embeddings degrades gracefully on API errors (e.g. quota
    # exhaustion), so a configured key doesn't guarantee the cache exists.
    METHODS.append("openai")

embeddings_by_method = {}
labels_by_method = {}
for method in METHODS:
    method_suffix = roberta_suffix if method == "roberta" else suffix
    arr = load_cached(f"{method}_train_{method_suffix}")
    assert arr is not None, f"Missing cached embeddings for '{method}' — run 01_embeddings.ipynb first"
    embeddings_by_method[method] = arr
    labels_by_method[method] = roberta_true_labels if method == "roberta" else true_labels
    assert arr.shape[0] == len(labels_by_method[method]), \
        f"{method}: embeddings ({arr.shape[0]} rows) and labels ({len(labels_by_method[method])} rows) size mismatch"

In [3]:
config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = {}

for method, emb in embeddings_by_method.items():
    method_labels = labels_by_method[method]

    reducer = umap.UMAP(n_components=50, metric="cosine", random_state=config.SEED)
    emb_reduced = reducer.fit_transform(emb)

    kmeans = KMeans(n_clusters=config.NUM_CLASSES, random_state=config.SEED, n_init=10)
    km_labels = kmeans.fit_predict(emb_reduced)
    km_metrics = evaluate_unsupervised(method_labels, km_labels, emb_reduced)
    all_results[f"{method}_kmeans"] = km_metrics

    clusterer = hdbscan.HDBSCAN(min_cluster_size=50, metric="euclidean",
                                 cluster_selection_method="eom")
    hdb_labels = clusterer.fit_predict(emb_reduced)
    hdb_metrics = evaluate_unsupervised(method_labels, hdb_labels, emb_reduced)
    all_results[f"{method}_hdbscan"] = hdb_metrics

    print(f"{method}: KMeans ACC={km_metrics['ACC (Hungarian)']:.3f} | "
          f"HDBSCAN coverage={hdb_metrics['Coverage']:.2f} ACC={hdb_metrics['ACC (Hungarian)']:.3f}")

for name, metrics in all_results.items():
    with open(config.RESULTS_DIR / f"metrics_{name}.json", "w") as f:
        json.dump(metrics, f, indent=2)

print(f"Saved {len(all_results)} result files to {config.RESULTS_DIR}")

C:\Users\ACER\OneDrive\Documents\final-project\.worktrees\autolabel-notebooks\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


tfidf: KMeans ACC=0.808 | HDBSCAN coverage=0.96 ACC=0.262


C:\Users\ACER\OneDrive\Documents\final-project\.worktrees\autolabel-notebooks\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


minilm: KMeans ACC=0.830 | HDBSCAN coverage=1.00 ACC=0.493


C:\Users\ACER\OneDrive\Documents\final-project\.worktrees\autolabel-notebooks\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


roberta: KMeans ACC=0.518 | HDBSCAN coverage=0.00 ACC=0.000
Saved 6 result files to C:\Users\ACER\OneDrive\Documents\final-project\.worktrees\autolabel-notebooks\results


### Optional stretch: LLM cluster naming

Instead of majority-vote mapping (which uses hidden ground truth and is
eval-only), an LLM could name each cluster from its top-N nearest documents
via `openai_client.chat.completions.create(...)`, following the
`llm_name_cluster` pattern in `autolabel_project_spec.md`. Not required for
the core comparison table — skipped here.